In [1]:
from deepfix_sdk import DeepFixClient
import os

d:\workspace\repos\deepfix\.venv\Lib\site-packages\deepchecks\core\serialization\dataframe\html.py:16: UserWarning:

pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.



In [2]:
os.environ["DEEPFIX_API_KEY"] = "DEEPFIX-IS-AMAZING"

In [3]:
client = DeepFixClient(api_url="http://deepfix.delcaux.com", timeout=120)

# Tabular data

## Classification

In [4]:
from deepfix_sdk.zoo.datasets import load_adult_classification
from deepfix_sdk.data.datasets import TabularDataset

In [5]:
train, test = load_adult_classification(as_train_test=True)
dataset_name = "adult-classification"

label = "income"
cat_features = train.select_dtypes(include=['object','string','category']).columns.tolist()
cat_features.remove(label)

train_data = TabularDataset(dataset=train, dataset_name=dataset_name, label=label, cat_features=cat_features)
val_data = TabularDataset(dataset=test, dataset_name=dataset_name, label=label, cat_features=cat_features)

In [6]:
train.head()

,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K


In [ ]:
client.ingest_dataset(
    dataset_name=dataset_name,
    data_type="tabular",
    train_data=train_data,
    test_data=val_data,
    train_test_validation=True,
    data_integrity=True,
    batch_size=8,
    overwrite=True,
)

In [7]:
# Diagnose dataset
result = client.diagnose_dataset(dataset_name=dataset_name)

Output()

✓ Analysis complete!

In [10]:
# Visualize results
result.to_text(verbose=False)

╭──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                               DEEPFIX ANALYSIS RESULT                                                │
╰──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────────────── Summary ───────────────────────────────────────────────────────╮
│ The integrated analysis reveals critical data quality issues requiring immediate attention, particularly the         │
│ significant label drift between train and test sets (Cramer's V=0.22) and high toxicity outlier ratios (16-16.43%).  │
│ While overall data hygiene is strong with minimal duplicates and no leakage, the tokenizer compatibility issues and  │
│ dataset artifact format problems indicate technical debt. Priority should be given to addressing the label           │
│ distribution mismatch and data quality outliers to ensure model reliability, followed by resolving technical         │
│ compatibility issues for sustained analytics effectiveness.                                                          │
╰──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

                                      Summary Statistics                                      
 Metric                          Value                                                        
 Total Findings                  4                                                            
 Severity Distribution           HIGH: 2  MEDIUM: 2                                           

                                  HIGH Severity Issues (2)                                   
┏━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ #   ┃ Finding                                  ┃ Action                                   ┃
┡━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ 1   │ Critical label distribution mismatch     │ Implement stratified sampling or         │
│     │ between training and test datasets       │ re-split datasets to ensure balanced     │
│     │ Evidence: Label Drift check failed with  │ label representation across train/test   │
│     │ Cramer's V score of 0.22 exceeding the   │ sets                                     │
│     │ 0.15 threshold, indicating significant   │ Label distribution drift undermines      │
│     │ distribution differences                 │ model validation reliability and         │
│     │                                          │ generalization capabilities              │
│ 2   │ Elevated toxicity property outliers      │ Conduct manual review of toxicity        │
│     │ suggesting data quality issues           │ outliers and implement data cleaning     │
│     │ Evidence: Text Property Outliers check   │ procedures                               │
│     │ identified 16-16.43% outlier ratio in    │ High outlier ratios may indicate         │
│     │ toxicity measurements, significantly     │ measurement errors or genuine extreme    │
│     │ above the 5% acceptable threshold        │ values requiring special handling        │
└─────┴──────────────────────────────────────────┴──────────────────────────────────────────┘

                                 MEDIUM Severity Issues (2)                                  
┏━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ #   ┃ Finding                                  ┃ Action                                   ┃
┡━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ 1   │ Tokenizer vocabulary gaps affecting text │ Expand tokenizer vocabulary or implement │
│     │ processing                               │ text preprocessing to handle             │
│     │ Evidence: Unknown Tokens check revealed  │ out-of-vocabulary tokens                 │
│     │ 0.68-0.79% of tokens unsupported by      │ Unknown tokens lead to information loss  │
│     │ current tokenizer                        │ and reduced model performance on diverse │
│     │                                          │ text inputs                              │
│ 2   │ Dataset artifact structure compatibility │ Validate dataset artifact structure and  │
│     │ issue with analysis tools                │ update analysis tools to handle current  │
│     │ Evidence: DatasetArtifactsAnalyzer       │ artifact format                          │
│     │ failed due to missing 'statistics'       │ Analysis tool compatibility is essential │
│     │ attribute, indicating potential artifact │ for comprehensive dataset evaluation and │
│     │ format mismatch                          │ quality assessment                       │
└─────┴──────────────────────────────────────────┴──────────────────────────────────────────┘

''

# Computer vision

## Image classification

In [11]:
from deepfix_sdk.zoo.datasets.foodwaste import load_train_and_val_datasets
from deepfix_sdk.data.datasets import ImageClassificationDataset

In [12]:
dataset_name = "cafetaria-foodwaste-lstroetmann"
# Load image datasets
train_data, val_data = load_train_and_val_datasets(
    image_size=448,
    batch_size=8,
    num_workers=4,
    pin_memory=False,
)
train_data = ImageClassificationDataset(dataset_name=dataset_name, dataset=train_data)
val_data = ImageClassificationDataset(dataset_name=dataset_name, dataset=val_data)

Getting label mapping: 100%|██████████| 375/375 [00:03<00:00, 107.85it/s]


In [13]:
client.ingest_dataset(
    dataset_name=dataset_name,
    data_type="vision",
    train_data=train_data,
    test_data=val_data,
    train_test_validation=val_data is not None,
    data_integrity=True,
    batch_size=8,
    overwrite=True,
)

Computing dataset base statistics: 100%|██████████| 160/160 [00:11<00:00, 14.06it/s]


In [14]:
# Diagnose dataset
result = client.diagnose_dataset(dataset_name=dataset_name)

Output()

✓ Analysis complete!

In [15]:
# Visualize results
result.to_text()

╭──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                               DEEPFIX ANALYSIS RESULT                                                │
╰──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────────────── Summary ───────────────────────────────────────────────────────╮
│ The cross-artifact analysis reveals catastrophic data quality issues that invalidate the current machine learning    │
│ setup. The test set suffers from severe label distribution drift (Cramer's V=0.92) and contains 75% new labels not   │
│ seen in training, indicating fundamental problems with data partitioning. Additionally, significant differences in   │
│ image properties suggest inconsistent acquisition conditions. These issues collectively mean that any model          │
│ evaluation would be unreliable. Immediate remediation of the data splitting methodology and image standardization is │
│ required before proceeding with model development.                                                                   │
╰──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

                                      Summary Statistics                                      
 Metric                          Value                                                        
 Total Findings                  3                                                            
 Severity Distribution           HIGH: 2  MEDIUM: 1                                           

                                  HIGH Severity Issues (2)                                   
┏━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ #   ┃ Finding                                  ┃ Action                                   ┃
┡━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ 1   │ Critical data partitioning failure       │ Immediately halt model development and   │
│     │ causing unrepresentative test set        │ recreate the train-test split using      │
│     │ Evidence: Combined evidence from         │ proper stratified sampling techniques    │
│     │ Deepchecks: Label drift check failed     │ The test set is fundamentally invalid    │
│     │ with Cramer's V score of 0.92 (far       │ for evaluation due to severe label       │
│     │ exceeding 0.15 threshold) and 75% of     │ distribution mismatch and leakage,       │
│     │ test set labels were not present in      │ making any model performance metrics     │
│     │ training                                 │ meaningless                              │
│ 2   │ Systematic differences in image          │ Standardize image collection protocols   │
│     │ acquisition conditions between datasets  │ and apply normalization techniques to    │
│     │ Evidence: Multiple image property drift  │ align visual characteristics             │
│     │ failures: Brightness (KS=0.42), RMS      │ Large differences in brightness,         │
│     │ Contrast (KS=0.5), Red Intensity         │ contrast, and color properties will      │
│     │ (KS=0.83), Green Intensity (KS=0.82),    │ cause models to learn dataset-specific   │
│     │ Blue Intensity (KS=0.96)                 │ artifacts rather than generalizable      │
│     │                                          │ features                                 │
└─────┴──────────────────────────────────────────┴──────────────────────────────────────────┘

                                 MEDIUM Severity Issues (1)                                  
┏━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ #   ┃ Finding                                  ┃ Action                                   ┃
┡━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ 1   │ Incomplete data quality assessment       │ Implement comprehensive data validation  │
│     │ framework                                │ pipeline including outlier detection,    │
│     │ Evidence: DatasetArtifactsAnalyzer       │ label consistency checks, and metadata   │
│     │ failed due to technical issues, and      │ validation                               │
│     │ Deepchecks data integrity section was    │ Current assessment gaps prevent          │
│     │ incomplete                               │ identification of additional data        │
│     │                                          │ quality issues that could impact model   │
│     │                                          │ reliability and performance              │
└─────┴──────────────────────────────────────────┴──────────────────────────────────────────┘

''

## Object detection

In [16]:
from deepfix_sdk.data.datasets import ObjectDetectionDataset

In [17]:
dataset_name = "general_dataset"
train_data = ObjectDetectionDataset.from_coco(
    dataset_name=dataset_name,
    images_directory_path=r"D:\workspace\general_dataset\coco\train",
    annotations_path=r"D:\workspace\general_dataset\coco\annotations\annotations_train.json",
)
val_data = ObjectDetectionDataset.from_coco(
    dataset_name=dataset_name,
    images_directory_path=r"D:\workspace\general_dataset\coco\val",
    annotations_path=r"D:\workspace\general_dataset\coco\annotations\annotations_val.json",
)

In [18]:
client.ingest_dataset(
    dataset_name=dataset_name,
    data_type="vision",
    train_data=train_data,
    test_data=val_data,
    train_test_validation=val_data is not None,
    data_integrity=True,
    batch_size=8,
    overwrite=True,
)

Computing base box statistics: 100%|██████████| 668/668 [00:00<?, ?it/s]


In [19]:
# Diagnose dataset
result = client.diagnose_dataset(dataset_name=dataset_name)

Output()

✓ Analysis complete!

In [20]:
# Visualize results
result.to_text()

╭──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                               DEEPFIX ANALYSIS RESULT                                                │
╰──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────────────── Summary ───────────────────────────────────────────────────────╮
│ The consolidated analysis indicates that the primary risk of overfitting stems from data quality issues identified   │
│ by the Deepchecks analyzer, specifically an unstable feature-label relationship and significant brightness drift     │
│ between train and test sets. These issues suggest the model may overfit to spurious correlations and specific        │
│ lighting conditions. The failure of the DatasetArtifacts analyzer and the incomplete data integrity assessment in    │
│ the Deepchecks results mean that the overall data quality picture is incomplete, presenting a secondary,             │
│ medium-severity risk. Recommendations focus on addressing the identified instabilities and completing the missing    │
│ data integrity checks to ensure the model learns robust, generalizable patterns.                                     │
╰──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

                                      Summary Statistics                                      
 Metric                          Value                                                        
 Total Findings                  2                                                            
 Severity Distribution           HIGH: 1  MEDIUM: 1                                           

                                  HIGH Severity Issues (1)                                   
┏━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ #   ┃ Finding                                  ┃ Action                                   ┃
┡━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ 1   │ Critical data quality and stability      │ Prioritize mitigating the identified     │
│     │ issues identified as primary overfitting │ feature-label instability and brightness │
│     │ risks                                    │ drift. Investigate the technical error   │
│     │ Evidence: Deepchecks analysis revealed a │ to enable a full DatasetArtifacts        │
│     │ high-severity unstable feature-label     │ analysis for a comprehensive view.       │
│     │ relationship (PPS difference: 0.21) and  │ The model is at high risk of overfitting │
│     │ significant image brightness drift (KS   │ due to learning non-generalizable        │
│     │ score: 0.29). The DatasetArtifacts       │ correlations and being sensitive to      │
│     │ analysis was unavailable due to a system │ lighting variations. A complete dataset  │
│     │ error, preventing a complete dataset     │ analysis is needed to rule out other     │
│     │ assessment.                              │ underlying data issues.                  │
└─────┴──────────────────────────────────────────┴──────────────────────────────────────────┘

                                 MEDIUM Severity Issues (1)                                  
┏━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ #   ┃ Finding                                  ┃ Action                                   ┃
┡━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ 1   │ Incomplete data integrity validation     │ Run a full suite of data integrity       │
│     │ obscures potential data quality issues   │ checks, including outlier detection and  │
│     │ Evidence: The Deepchecks analysis noted  │ label validation, to identify any hidden │
│     │ an empty or missing data integrity       │ data quality problems.                   │
│     │ section, leaving outlier detection and   │ Unassessed data integrity issues can     │
│     │ label consistency unassessed. Combined   │ silently contribute to overfitting. A    │
│     │ with the failed DatasetArtifacts         │ complete assessment is crucial for       │
│     │ analysis, there is a gap in the overall  │ building a robust model.                 │
│     │ data quality evaluation.                 │                                          │
└─────┴──────────────────────────────────────────┴──────────────────────────────────────────┘

''

## Semantic segmentation

In [21]:
from deepfix_sdk.data.datasets import SemanticSegmentationDataset
from deepfix_sdk.zoo.datasets import load_segmentation_dataset

In [22]:
dataset_name = "coco_segmentation"
train_data, val_data = load_segmentation_dataset(
    batch_size=8,
    shuffle=False,
    pin_memory=False,
)
train_data = SemanticSegmentationDataset(dataset_name=dataset_name, dataset=train_data.dataset)
val_data = SemanticSegmentationDataset(dataset_name=dataset_name, dataset=val_data.dataset)


In [23]:
client.ingest_dataset(
    dataset_name=dataset_name,
    data_type="vision",
    train_data=train_data,
    test_data=val_data,
    train_test_validation=True,
    data_integrity=True,
    batch_size=8,
    overwrite=True,
)

Computing dataset base statistics: 100%|██████████| 49/49 [00:03<00:00, 14.83it/s]


In [24]:
# Diagnose dataset
result = client.diagnose_dataset(dataset_name=dataset_name)

Output()

✓ Analysis complete!

In [25]:
# Visualize results
result.to_text()

╭──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                               DEEPFIX ANALYSIS RESULT                                                │
╰──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────────────── Summary ───────────────────────────────────────────────────────╮
│ The analysis reveals critical data quality issues primarily identified through Deepchecks validation. The most       │
│ severe problems involve significant distribution mismatches between training and test datasets, including class      │
│ imbalance (0.17 categorical drift) and color property differences (red: 0.2, green: 0.24 drift scores). These        │
│ distribution inconsistencies threaten model reliability and generalization. Additionally, gaps in the data           │
│ validation framework (evidenced by incomplete integrity checks and analyzer failures) suggest a need for stronger    │
│ quality assurance processes. Immediate attention should focus on rebalancing datasets and implementing comprehensive │
│ validation to ensure model performance reflects true capabilities rather than dataset artifacts.                     │
╰──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

                                      Summary Statistics                                      
 Metric                          Value                                                        
 Total Findings                  2                                                            
 Severity Distribution           HIGH: 1  MEDIUM: 1                                           

                                  HIGH Severity Issues (1)                                   
┏━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ #   ┃ Finding                                  ┃ Action                                   ┃
┡━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ 1   │ Critical data distribution               │ Implement comprehensive data             │
│     │ inconsistencies between training and     │ distribution analysis and rebalance      │
│     │ validation sets                          │ training/validation splits to ensure     │
│     │ Evidence: Deepchecks analysis shows      │ consistent class and feature             │
│     │ categorical drift score of 0.17 for      │ distributions                            │
│     │ 'Samples Per Class' (exceeding 0.15      │ Distribution mismatches between datasets │
│     │ threshold) and color property drifts     │ lead to unreliable model evaluation and  │
│     │ (Mean Red: 0.2, Mean Green: 0.24         │ poor generalization to real-world data   │
│     │ exceeding 0.2 threshold), indicating     │                                          │
│     │ significant distribution mismatches      │                                          │
└─────┴──────────────────────────────────────────┴──────────────────────────────────────────┘

                                 MEDIUM Severity Issues (1)                                  
┏━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ #   ┃ Finding                                  ┃ Action                                   ┃
┡━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ 1   │ Insufficient data quality validation     │ Establish robust data validation         │
│     │ framework                                │ pipeline with comprehensive integrity    │
│     │ Evidence: Deepchecks analysis indicates  │ checks, outlier detection, and automated │
│     │ incomplete data integrity validation     │ quality monitoring                       │
│     │ section, and DatasetArtifactsAnalyzer    │ Missing or incomplete validation         │
│     │ failed due to technical issues,          │ increases risk of undetected data        │
│     │ suggesting gaps in the data validation   │ quality issues that can compromise model │
│     │ pipeline                                 │ performance and reliability              │
└─────┴──────────────────────────────────────────┴──────────────────────────────────────────┘

''

# NLP

## Sentence classification

In [26]:
from deepfix_sdk.zoo.datasets import load_tweet_emotion_classification
from deepfix_sdk.data.datasets import NLPDataset

In [27]:
train_data, test_data = load_tweet_emotion_classification(
        as_train_test=True, include_embeddings=True
    )
dataset_name = "tweet_emotion_classification"
train_data = NLPDataset(dataset_name=dataset_name, dataset=train_data)
test_data = NLPDataset(dataset_name=dataset_name, dataset=test_data)

In [28]:
client.ingest_dataset(
    dataset_name=dataset_name,
    data_type="nlp",
    train_data=train_data,
    test_data=test_data,
    train_test_validation=True,
    data_integrity=True,
    batch_size=8,
    overwrite=True,
)

deepchecks - WARNING - Could not find model's classes, using the observed classes. In order to make sure the classes used by the model are inferred correctly, please use the model_classes argument


In [29]:
# Diagnose dataset
result = client.diagnose_dataset(dataset_name=dataset_name)

Output()

✓ Analysis complete!

In [30]:
# Visualize results
result.to_text()

╭──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                               DEEPFIX ANALYSIS RESULT                                                │
╰──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────────────── Summary ───────────────────────────────────────────────────────╮
│ The cross-artifact analysis was partially successful. The DatasetArtifactsAnalyzer encountered a technical error and │
│ could not provide results. However, the DeepchecksArtifactsAnalyzer identified critical data quality issues. The     │
│ most severe finding is a high-confidence label distribution drift between train and test sets, which poses a         │
│ significant risk to model validity. Additional concerns include a high ratio of outliers in text toxicity and the    │
│ presence of unknown tokens, both rated as medium severity. A low-severity text embeddings drift was also noted.      │
│ Recommendations focus on data preprocessing, tokenizer updates, and careful performance monitoring to ensure model   │
│ robustness. The failure of one analyzer highlights a potential need to review the artifact analysis pipeline for     │
│ stability.                                                                                                           │
╰──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

                                      Summary Statistics                                      
 Metric                          Value                                                        
 Total Findings                  4                                                            
 Severity Distribution           MEDIUM: 2  HIGH: 1  LOW: 1                                   

                                  HIGH Severity Issues (1)                                   
┏━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ #   ┃ Finding                                  ┃ Action                                   ┃
┡━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ 1   │ Significant label distribution drift     │ Re-examine the train-test split          │
│     │ between train and test sets              │ methodology to ensure proper             │
│     │ Evidence: Label drift check failed with  │ stratification                           │
│     │ Cramer's V score of 0.22, exceeding the  │ Label distribution mismatch can lead to  │
│     │ 0.15 threshold, indicating substantial   │ unreliable performance metrics and model │
│     │ distribution shift in emotion labels     │ overfitting to train-specific patterns   │
└─────┴──────────────────────────────────────────┴──────────────────────────────────────────┘

                                 MEDIUM Severity Issues (2)                                  
┏━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ #   ┃ Finding                                  ┃ Action                                   ┃
┡━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ 1   │ High outlier ratio in text properties,   │ Investigate and clean outliers in the    │
│     │ particularly Toxicity                    │ Toxicity property, or implement robust   │
│     │ Evidence: Text property outliers check   │ preprocessing                            │
│     │ failed with Toxicity property showing    │ High outlier ratios can distort feature  │
│     │ 16.43% outlier ratio, significantly      │ relationships and model training,        │
│     │ above the 5% threshold                   │ potentially leading to poor              │
│     │                                          │ generalization                           │
│ 2   │ Presence of unknown tokens indicating    │ Update tokenizer vocabulary or           │
│     │ tokenizer coverage gaps                  │ preprocess text to handle unknown tokens │
│     │ Evidence: Unknown tokens check failed    │ appropriately                            │
│     │ with ratios of 0.79% and 0.68%,          │ Unknown tokens can degrade model         │
│     │ indicating unsupported tokens in the     │ performance and introduce noise in text  │
│     │ dataset                                  │ representations                          │
└─────┴──────────────────────────────────────────┴──────────────────────────────────────────┘

                                   LOW Severity Issues (1)                                   
┏━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ #   ┃ Finding                                  ┃ Action                                   ┃
┡━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ 1   │ Moderate text embeddings drift           │ Monitor model performance closely and    │
│     │ suggesting domain shift                  │ consider domain adaptation techniques if │
│     │ Evidence: Text embeddings drift showed   │ performance degrades                     │
│     │ AUC of 0.6, indicating some domain shift │ Domain shift can affect model            │
│     │ between train and test distributions     │ generalization, though the current level │
│     │                                          │ may be acceptable for deployment         │
└─────┴──────────────────────────────────────────┴──────────────────────────────────────────┘

''